In [7]:
import os
import time
import pandas as pd
import requests
from tqdm import tqdm
import numpy as np
from datetime import datetime

In [8]:
os.chdir(r"D:\spinny_project")
print("Current folder:", os.getcwd())

Current folder: D:\spinny_project


In [9]:
base_url = "https://api.spinny.com/v3/api/listing/v6/"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Origin": "https://www.spinny.com",
    "Referer": "https://www.spinny.com/used-cars-in-delhi-ncr/s/",
    "platform": "web"
}

In [10]:
cities = {
    "delhi-ncr": "Delhi NCR",
    "bangalore": "Bangalore",
    "hyderabad": "Hyderabad",
    "mumbai": "Mumbai",
    "pune": "Pune",
    "delhi": "Delhi",
    "gurgaon": "Gurgaon",
    "noida": "Noida",
    "ahmedabad": "Ahmedabad",
    "chennai": "Chennai",
    "kolkata": "Kolkata",
    "lucknow": "Lucknow",
    "jaipur": "Jaipur",
    "chandigarh": "Chandigarh",
    "agra": "Agra",
    "ambala": "Ambala",
    "coimbatore": "Coimbatore",
    "faridabad": "Faridabad",
    "ghaziabad": "Ghaziabad",
    "jodhpur": "Jodhpur",
    "kanpur": "Kanpur",
    "karnal": "Karnal",
    "kochi": "Kochi",
    "mysuru": "Mysuru",
    "prayagraj": "Prayagraj",
    "sonipat": "Sonipat",
    "vadodara": "Vadodara",
    "visakhapatnam": "Visakhapatnam",
}

In [11]:
def safe_get(d, key, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return default

def extract_car(car):
    discount = safe_get(car, "discount", {})
    discount_v3 = safe_get(car, "discount_v3", {})
    price_breakdown = safe_get(car, "price_breakdown", {})
    price_breakdown_v2 = safe_get(car, "price_breakdown_v2", {})
    current_price_data = safe_get(car, "current_price_data", {})
    insurance_validity = safe_get(car, "applicable_insurance_validity", {})

    return {
        "car_id": safe_get(car, "id"),
        "city": safe_get(car, "city"),
        "make": safe_get(car, "make"),
        "model": safe_get(car, "model"),
        "variant": safe_get(car, "variant"),
        "make_year": safe_get(car, "make_year"),
        "registration_year": safe_get(car, "registration_year"),
        "price": safe_get(car, "price"),
        "emi": safe_get(car, "emi"),
        "mileage": safe_get(car, "mileage"),
        "round_off_mileage": safe_get(car, "round_off_mileage"),
        "fuel_type": safe_get(car, "fuel_type"),
        "no_of_owners": safe_get(car, "no_of_owners"),
        "transmission": safe_get(car, "transmission"),
        "transmission_sub_type": safe_get(car, "transmission_sub_type"),
        "rto": safe_get(car, "rto"),
        "hub": safe_get(car, "hub"),
        "hub_short_name": safe_get(car, "hub_short_name"),
        "seller_type": safe_get(car, "seller_type"),
        "body_type": safe_get(car, "body_type"),
        "car_category": safe_get(car, "car_category"),
        "procurement_category": safe_get(car, "procurement_category"),
        "tag_status": safe_get(car, "tag_status"),
        "permanent_url": safe_get(car, "permanent_url"),
        "token_amount": safe_get(car, "token_amount"),
        "booked": safe_get(car, "booked"),
        "sold": safe_get(car, "sold"),
        "upcoming": safe_get(car, "upcoming"),
        "is_fixed_price": safe_get(car, "is_fixed_price"),
        "home_test_drive_available": safe_get(car, "home_test_drive_available"),
        "showroom_visit_available": safe_get(car, "showroom_visit_available"),
        "has_three_sixty_view": safe_get(car, "has_three_sixty_view"),
        "color": safe_get(car, "color"),
        "shortlist_count": safe_get(car, "shortlist_count"),
        "exchange_bonus": safe_get(car, "exchange_bonus"),
        "is_assured_plus": safe_get(car, "is_assured_plus"),
        "discount_value": safe_get(discount, "value"),
        "discount_end_time": safe_get(discount, "end_time"),
        "discount_name": safe_get(discount, "discount_name"),
        "discount_v3_value": safe_get(discount_v3, "value"),
        "applicable_insurance_year": safe_get(insurance_validity, "year"),
        "applicable_insurance_month": safe_get(insurance_validity, "month"),
        "applicable_insurance_day": safe_get(insurance_validity, "day"),
        "base_listing_price": safe_get(price_breakdown, "base_listing_price"),
        "listing_price": safe_get(price_breakdown, "listing_price"),
        "listing_price_v2": safe_get(price_breakdown_v2, "listing_price"),
        "original_price_v2": safe_get(price_breakdown_v2, "original_price"),
        "adjusted_mid_listing_price": safe_get(current_price_data, "adjusted_mid_listing_price"),
        "current_listing_price": safe_get(current_price_data, "listing_price"),
        "image_count": len(safe_get(car, "images", [])),
    }

def fetch_city_listings(city_slug, city_label, sleep_seconds=0.3, max_pages=500):
    rows = []

    for page in range(1, max_pages + 1):
        params = {
            "city": city_slug,
            "product_type": "cars",
            "category": "used",
            "availability": "available,booked",
            "page": page,
            "show_max_on_assured": "true",
            "custom_budget_sort": "true",
            "prioritize_filter_listing": "true",
            "high_intent_required": "true",
            "active_banner": "true",
            "added_in_inventory": "true",
            "is_pulse_exp": "false",
            "is_new_price": "true"
        }

        resp = requests.get(base_url, params=params, headers=headers, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        results = data.get("results", [])
        if not results:
            print(f"{city_label} | page {page} | no results, stopping")
            break

        for car in results:
            row = extract_car(car)
            row["city_slug"] = city_slug
            row["city_label"] = city_label
            row["page"] = page
            rows.append(row)

        print(f"{city_label} | page {page} | rows so far: {len(rows)}")
        time.sleep(sleep_seconds)

    return pd.DataFrame(rows)

In [12]:
master_df_list = []

for city_slug, city_label in tqdm(cities.items()):
    df_city = fetch_city_listings(city_slug, city_label)
    master_df_list.append(df_city)

master_df = pd.concat(master_df_list, ignore_index=True)

print("Final shape:", master_df.shape)
print("Cities scraped:", master_df["city_label"].nunique())

  0%|                                                                                           | 0/28 [00:00<?, ?it/s]

Delhi NCR | page 1 | rows so far: 20
Delhi NCR | page 2 | rows so far: 40
Delhi NCR | page 3 | rows so far: 60
Delhi NCR | page 4 | rows so far: 80
Delhi NCR | page 5 | rows so far: 100
Delhi NCR | page 6 | rows so far: 120
Delhi NCR | page 7 | rows so far: 140
Delhi NCR | page 8 | rows so far: 157
Delhi NCR | page 9 | rows so far: 173
Delhi NCR | page 10 | rows so far: 189
Delhi NCR | page 11 | rows so far: 205
Delhi NCR | page 12 | rows so far: 221
Delhi NCR | page 13 | rows so far: 237
Delhi NCR | page 14 | rows so far: 253
Delhi NCR | page 15 | rows so far: 269
Delhi NCR | page 16 | rows so far: 285
Delhi NCR | page 17 | rows so far: 301
Delhi NCR | page 18 | rows so far: 317
Delhi NCR | page 19 | rows so far: 333
Delhi NCR | page 20 | rows so far: 349
Delhi NCR | page 21 | rows so far: 365
Delhi NCR | page 22 | rows so far: 381
Delhi NCR | page 23 | rows so far: 397
Delhi NCR | page 24 | rows so far: 413
Delhi NCR | page 25 | rows so far: 429
Delhi NCR | page 26 | rows so far: 445

  4%|██▉                                                                                | 1/28 [01:26<39:05, 86.87s/it]

Delhi NCR | page 80 | no results, stopping
Bangalore | page 1 | rows so far: 20
Bangalore | page 2 | rows so far: 40
Bangalore | page 3 | rows so far: 60
Bangalore | page 4 | rows so far: 79
Bangalore | page 5 | rows so far: 94
Bangalore | page 6 | rows so far: 109
Bangalore | page 7 | rows so far: 124
Bangalore | page 8 | rows so far: 139
Bangalore | page 9 | rows so far: 154
Bangalore | page 10 | rows so far: 169
Bangalore | page 11 | rows so far: 184
Bangalore | page 12 | rows so far: 199
Bangalore | page 13 | rows so far: 214
Bangalore | page 14 | rows so far: 229
Bangalore | page 15 | rows so far: 244
Bangalore | page 16 | rows so far: 259
Bangalore | page 17 | rows so far: 274
Bangalore | page 18 | rows so far: 289
Bangalore | page 19 | rows so far: 304
Bangalore | page 20 | rows so far: 319
Bangalore | page 21 | rows so far: 334
Bangalore | page 22 | rows so far: 349
Bangalore | page 23 | rows so far: 364
Bangalore | page 24 | rows so far: 379
Bangalore | page 25 | rows so far: 

  7%|█████▉                                                                             | 2/28 [02:14<27:44, 64.00s/it]

...

Sonipat | page 3 | no results, stopping
Vadodara | page 1 | rows so far: 20
Vadodara | page 2 | rows so far: 33


 96%|███████████████████████████████████████████████████████████████████████████████   | 27/28 [09:51<00:03,  3.74s/it]

Vadodara | page 3 | no results, stopping
Visakhapatnam | page 1 | rows so far: 20
Visakhapatnam | page 2 | rows so far: 40
Visakhapatnam | page 3 | rows so far: 60
Visakhapatnam | page 4 | rows so far: 67


100%|██████████████████████████████████████████████████████████████████████████████████| 28/28 [09:56<00:00, 21.31s/it]

Visakhapatnam | page 5 | no results, stopping
Final shape: (7709, 53)
Cities scraped: 28


In [13]:
print("Final shape:", master_df.shape)
print("Cities scraped:", master_df["city_label"].nunique())

Final shape: (7709, 53)
Cities scraped: 28


In [14]:
master_df["city_label"].value_counts().sort_index()

city_label
Agra               38
Ahmedabad         358
Ambala             26
Bangalore         629
Chandigarh        227
Chennai           263
Coimbatore         98
Delhi             697
Delhi NCR        1272
Faridabad         118
Ghaziabad         298
Gurgaon           311
Hyderabad         590
Jaipur            219
Jodhpur            39
Kanpur             59
Karnal             29
Kochi              84
Kolkata           293
Lucknow           269
Mumbai            467
Mysuru             45
Noida             476
Prayagraj          30
Pune              645
Sonipat            29
Vadodara           33
Visakhapatnam      67
Name: count, dtype: int64

In [15]:
import os

os.makedirs(r"D:\spinny_project\data\raw", exist_ok=True)

master_df.to_csv(
    r"D:\spinny_project\data\raw\spinny_master_raw.csv",
    index=False
)

print("Raw dataset saved successfully!")

Raw dataset saved successfully!
